In [1]:
import db_queries
from get_draws.api import get_draws
import pandas as pd, numpy as np

In [2]:
index_cols=['location_id','sex_id','age_group_id']

age_group_ids = [2,3,388,389,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 30, 31, 32, 235]
sex_ids = [1,2]
coverage_levels = [0.8]
years = [2021,2025]

DRAWS = [f'draw_{i}' for i in range(500)] # NOTE: Some GBD 2021 things return 1,000 but others don't

In [18]:
mean_difference = 3 # TODO: Use our actual effect sizes

In [3]:
coverage_data = 'vehicle_coverage.csv'

In [4]:
location_ids = (list(
                pd.read_csv(coverage_data)
                .location_id
                .unique()))
location_ids

[163, 214]

In [5]:
coverage = pd.read_csv('vehicle_coverage.csv')
coverage = coverage[coverage.nutrient == 'iron'].drop(columns=["nutrient"])

In [6]:
baseline_coverage = coverage[coverage.value_description == 'percent of population eating fortified vehicle'].drop(columns=["value_description"])
baseline_coverage

,location_id,vehicle,value
0,163,rice,0.45
2,214,bouillon,0.45


In [7]:
# NOTE: We assume all coverage is effective at baseline
effective_baseline_coverage = baseline_coverage.copy()

In [8]:
baseline_coverage.set_index(["location_id", "vehicle"]).value 

location_id  vehicle 
163          rice        0.45
214          bouillon    0.45
Name: value, dtype: float64

In [9]:
coverage[coverage.value_description == 'percent of population eating industrially produced vehicle'].drop(columns=["value_description"]).set_index(["location_id", "vehicle"]).value

location_id  vehicle 
163          rice        0.75
214          bouillon    0.98
Name: value, dtype: float64

In [10]:
effective_counterfactual_coverage = (
    baseline_coverage.set_index(["location_id", "vehicle"]).value +
    # 80% of fortifiable fortified, 80% of that effective
    (coverage[coverage.value_description == 'percent of population eating industrially produced vehicle'].drop(columns=["value_description"]).set_index(["location_id", "vehicle"]).value - baseline_coverage.set_index(["location_id", "vehicle"]).value) * 0.8 * 0.8
).reset_index()
effective_counterfactual_coverage

,location_id,vehicle,value
0,163,rice,0.6420
1,214,bouillon,0.7892


In [11]:
def repeat_for_age_and_sex(df):
    return (
        df.assign(key=1)
            .merge(pd.DataFrame({'key': 1, 'age_group_id': age_group_ids}), on="key")
            .merge(pd.DataFrame({'key': 1, 'sex_id': sex_ids}), on='key')
            .drop(columns=['key'])
    )

In [12]:
effective_baseline_coverage = repeat_for_age_and_sex(effective_baseline_coverage).set_index(["location_id", "sex_id", "age_group_id", "vehicle"])
effective_counterfactual_coverage = repeat_for_age_and_sex(effective_counterfactual_coverage).set_index(["location_id", "sex_id", "age_group_id", "vehicle"])

In [13]:
effective_baseline_coverage

value
location_id sex_id age_group_id vehicle        
163         1      2            rice       0.45
            2      2            rice       0.45
            1      3            rice       0.45
            2      3            rice       0.45
            1      388          rice       0.45
...                                         ...
214         2      31           bouillon   0.45
            1      32           bouillon   0.45
            2      32           bouillon   0.45
            1      235          bouillon   0.45
            2      235          bouillon   0.45

[92 rows x 1 columns]

In [14]:
delta_effective_coverage = effective_counterfactual_coverage - effective_baseline_coverage
delta_effective_coverage

value
location_id sex_id age_group_id vehicle         
163         1      2            rice      0.1920
            2      2            rice      0.1920
            1      3            rice      0.1920
            2      3            rice      0.1920
            1      388          rice      0.1920
...                                          ...
214         2      31           bouillon  0.3392
            1      32           bouillon  0.3392
            2      32           bouillon  0.3392
            1      235          bouillon  0.3392
            2      235          bouillon  0.3392

[92 rows x 1 columns]

In [15]:
hgb_mean = (
    get_draws('modelable_entity_id',
                    10487,
                    source='epi',
                    location_id=location_ids,
                    age_group_id=age_group_ids,
                    sex_id=sex_ids,
                    year_id=2021,
                    release_id=9)
    .set_index(['location_id','sex_id','age_group_id'])
)
hgb_mean = hgb_mean[DRAWS].copy()

In [16]:
hgb_mean = hgb_mean.reset_index().assign(vehicle=lambda x: x.location_id.map({163: 'rice', 214: 'bouillon'})).set_index(list(hgb_mean.index.names) + ['vehicle'])
hgb_mean

draw_0      draw_1      draw_2  \
location_id sex_id age_group_id vehicle                                        
163         2      2            rice      141.027303  141.174892  143.047093   
                   3            rice      128.185915  129.096397  129.130806   
                   6            rice      112.932920  111.768831  112.598540   
                   7            rice      115.889117  117.854010  116.651884   
                   8            rice      117.312802  117.869717  117.696658   
...                                              ...         ...         ...   
214         2      31           bouillon  118.780034  118.418548  114.061451   
                   32           bouillon  109.477010  104.789317  110.610979   
                   235          bouillon  102.971577  100.970692   99.657313   
                   388          bouillon  104.152465  104.343166  100.933682   
                   389          bouillon  100.148802   99.844921   99.645127   

                                              draw_3      draw_4      draw_5  \
location_id sex_id age_group_id vehicle                                        
163         2      2            rice      142.497768  141.204487  141.832669   
                   3            rice      127.546092  129.886294  129.778156   
                   6            rice      114.147894  113.027698  112.048307   
                   7            rice      116.435834  117.758511  116.982931   
                   8            rice      118.290512  117.803942  117.556792   
...                                              ...         ...         ...   
214         2      31           bouillon  115.310723  116.190198  124.807436   
                   32           bouillon  108.944244  110.272233  102.837675   
                   235          bouillon   98.358658   96.036401  101.532534   
                   388          bouillon  103.672346  105.214707  102.017120   
                   389          bouillon  100.145850  100.608923  101.012091   

                                              draw_6      draw_7      draw_8  \
location_id sex_id age_group_id vehicle                                        
163         2      2            rice      140.942772  142.891759  141.388006   
                   3            rice      127.898826  129.079936  129.428546   
                   6            rice      112.435412  112.251242  114.540273   
                   7            rice      118.398544  117.928240  118.830370   
                   8            rice      116.771422  117.697237  117.938688   
...                                              ...         ...         ...   
214         2      31           bouillon  111.070989  119.262830  117.795365   
                   32           bouillon  110.397164  106.996263  113.492466   
                   235          bouillon  102.345743  101.051934  100.307373   
                   388          bouillon  101.666111  104.391111  101.872444   
                   389          bouillon  100.358797   98.995013   99.743129   

                                              draw_9  ...    draw_490  \
location_id sex_id age_group_id vehicle               ...               
163         2      2            rice      143.166841  ...  140.459189   
                   3            rice      128.289116  ...  128.716439   
                   6            rice      112.764552  ...  113.947008   
                   7            rice      117.453360  ...  115.670224   
                   8            rice      116.838995  ...  117.497554   
...                                              ...  ...         ...   
214         2      31           bouillon  126.255445  ...  117.542573   
                   32           bouillon  112.580471  ...  111.306577   
                   235          bouillon   98.663466  ...  101.142919   
                   388          bouillon  103.526095  ...  102.597493   
                   389          bouillon  100.8

In [19]:
counterfactual_hgb_mean = hgb_mean.add(delta_effective_coverage.value * mean_difference, axis=0)
counterfactual_hgb_mean

draw_0      draw_1      draw_2  \
location_id sex_id age_group_id vehicle                                        
163         1      2            rice      147.176654  149.679908  146.650897   
                   3            rice      129.169451  129.563080  132.140522   
                   6            rice      116.519763  117.149107  114.828198   
                   7            rice      126.437685  126.679005  124.307518   
                   8            rice      138.724186  139.772158  139.153766   
...                                              ...         ...         ...   
214         2      31           bouillon  119.797634  119.436148  115.079051   
                   32           bouillon  110.494610  105.806917  111.628579   
                   235          bouillon  103.989177  101.988292  100.674913   
                   388          bouillon  105.170065  105.360766  101.951282   
                   389          bouillon  101.166402  100.862521  100.662727   

                                              draw_3      draw_4      draw_5  \
location_id sex_id age_group_id vehicle                                        
163         1      2            rice      146.324027  150.083916  146.540431   
                   3            rice      129.622613  131.394723  129.591779   
                   6            rice      114.542774  115.171027  114.579191   
                   7            rice      126.429467  124.029578  125.896761   
                   8            rice      140.988738  140.399752  138.476611   
...                                              ...         ...         ...   
214         2      31           bouillon  116.328323  117.207798  125.825036   
                   32           bouillon  109.961844  111.289833  103.855275   
                   235          bouillon   99.376258   97.054001  102.550134   
                   388          bouillon  104.689946  106.232307  103.034720   
                   389          bouillon  101.163450  101.626523  102.029691   

                                              draw_6      draw_7      draw_8  \
location_id sex_id age_group_id vehicle                                        
163         1      2            rice      146.801025  145.892586  149.854837   
                   3            rice      131.710078  132.189459  131.090048   
                   6            rice      114.864712  115.732255  116.394374   
                   7            rice      126.668438  123.414715  127.895340   
                   8            rice      138.455467  140.022326  139.645579   
...                                              ...         ...         ...   
214         2      31           bouillon  112.088589  120.280430  118.812965   
                   32           bouillon  111.414764  108.013863  114.510066   
                   235          bouillon  103.363343  102.069534  101.324973   
                   388          bouillon  102.683711  105.408711  102.890044   
                   389          bouillon  101.376397  100.012613  100.760729   

                                              draw_9  ...    draw_490  \
location_id sex_id age_group_id vehicle               ...               
163         1      2            rice      148.023709  ...  148.907245   
                   3            rice      130.814115  ...  130.456855   
                   6            rice      115.344091  ...  115.725947   
                   7            rice      124.620043  ...  124.294214   
                   8            rice      140.108361  ...  140.269486   
...                                              ...  ...         ...   
214         2      31           bouillon  127.273045  ...  118.560173   
                   32           bouillon  113.598071  ...  112.324177   
                   235          bouillon   99.681066  ...  102.160519   
                   388          bouillon  104.543695  ...  103.615093   
                   389          bouillon  101.8

In [20]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("value").reset_index()

In [21]:
counterfactual_hgb_mean.columns.name = "draw"
counterfactual_hgb_mean = counterfactual_hgb_mean.stack().rename("value").reset_index()

In [23]:
hgb_sd = get_draws('modelable_entity_id',
                10488,
                source='epi',
                location_id=location_ids,
                age_group_id=age_group_ids,
                sex_id=sex_ids,
                year_id=2021,
                release_id=9)

hgb_sd = hgb_sd.set_index(['location_id','sex_id','age_group_id'])[DRAWS].copy()
hgb_sd

draw_0     draw_1     draw_2     draw_3  \
location_id sex_id age_group_id                                               
163         2      2             11.056572  15.419496  12.937057  14.108613   
                   3             18.143206  17.170122  18.638921  21.675732   
                   6             14.225182  14.168108  12.948419  12.913613   
                   7             13.709243  13.495770  11.389442  15.925441   
                   8             13.561928  15.733414  13.751380  15.264811   
...                                    ...        ...        ...        ...   
214         2      31            18.088001  20.580833   5.178689  16.104535   
                   32            12.785354  20.398206  15.979831  13.495828   
                   235           23.461384  26.340875  35.452001  34.646552   
                   388           22.493218  16.259081  17.452182  15.476258   
                   389           15.088749  16.763458  14.467460  14.968448   

                                    draw_4     draw_5     draw_6     draw_7  \
location_id sex_id age_group_id                                               
163         2      2             13.020095  12.577809  17.754828  13.099648   
                   3             17.362276  18.653702  20.182212  17.627105   
                   6             14.731795  14.601781  12.012659  11.527454   
                   7             16.126926  12.631853  16.717010  14.298076   
                   8             14.127997  14.013979  15.329002  15.397312   
...                                    ...        ...        ...        ...   
214         2      31            13.526887  29.906430   9.459181  12.848075   
                   32             9.890216  22.949436  11.103946  17.341702   
                   235           51.883828  28.301982  23.624796  24.595012   
                   388           20.107787  23.465305  22.878430  20.102228   
                   389           14.433290  15.700162  16.891376  14.383653   

                                    draw_8     draw_9  ...   draw_490  \
location_id sex_id age_group_id                        ...              
163         2      2             11.912721  13.534418  ...  10.329697   
                   3             18.478847  19.942071  ...  18.396669   
                   6             10.846907  13.933340  ...  12.666946   
                   7             15.839117  15.625613  ...  14.299624   
                   8             16.466307  11.996041  ...  14.884398   
...                                    ...        ...  ...        ...   
214         2      31            17.214625  24.926142  ...   9.460816   
                   32            10.731815   9.130296  ...   9.713881   
                   235           33.460763  31.062596  ...  30.098632   
                   388           19.558502  12.149664  ...  14.180015   
                   389           13.580970  13.668688  ...  13.607053   

                                  draw_491   draw_492   draw_493   draw_494  \
location_id sex_id age_group_id                                               
163         2      2             10.668878  13.046126  10.782160  11.950843   
                   3             19.537466  16.708525  20.642018  22.565014   
                   6             11.115035  11.686151  13.421744  13.699534   
                   7             14.091810  13.872200  17.720165  14.140329   
                   8             15.107632  13.951086  18.230387  13.800882   
...                                    ...        ...        ...        ...   
214         2      31            19.638156  13.887381  22.770907  11.876131   
                   32            17.133227  14.408574  24.032717  15.747865   
                   235           22.081338  49.638596  32.897283  36.931489   
                   388           14.757910  19.195717  24.988649  18.294609   
                   389           14.132132  15.332481  15.166399  15.459204   

  

In [24]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("value").reset_index()
hgb_sd

,location_id,sex_id,age_group_id,draw,value
0,163,2,2,draw_0,11.056572
1,163,2,2,draw_1,15.419496
2,163,2,2,draw_2,12.937057
3,163,2,2,draw_3,14.108613
4,163,2,2,draw_4,13.020095
...,...,...,...,...,...
45995,214,2,389,draw_495,13.412401
45996,214,2,389,draw_496,14.305163
45997,214,2,389,draw_497,15.815794
45998,214,2,389,draw_498,15.249249


In [25]:
import risk_distributions

In [26]:
mean_and_sd_hgb = pd.concat([
    hgb_mean.rename(columns={"value": "mean"}).merge(hgb_sd.rename(columns={"value": "sd"}), on=["location_id", "sex_id", "age_group_id", "draw"], how="outer").assign(scenario='baseline'),
    counterfactual_hgb_mean.rename(columns={"value": "mean"}).merge(hgb_sd.rename(columns={"value": "sd"}), on=["location_id", "sex_id", "age_group_id", "draw"], how="outer").assign(scenario='intervention')
])
mean_and_sd_hgb

,location_id,sex_id,age_group_id,vehicle,draw,mean,sd,scenario
0,163,2,2,rice,draw_0,141.027303,11.056572,baseline
1,163,2,2,rice,draw_1,141.174892,15.419496,baseline
2,163,2,2,rice,draw_2,143.047093,12.937057,baseline
3,163,2,2,rice,draw_3,142.497768,14.108613,baseline
4,163,2,2,rice,draw_4,141.204487,13.020095,baseline
...,...,...,...,...,...,...,...,...
45995,214,2,389,bouillon,draw_495,100.356215,13.412401,intervention
45996,214,2,389,bouillon,draw_496,100.399740,14.305163,intervention
45997,214,2,389,bouillon,draw_497,99.841120,15.815794,intervention
45998,214,2,389,bouillon,draw_498,100.311649,15.249249,intervention


In [27]:
mean_and_sd_hgb = mean_and_sd_hgb.assign(pregnant=0).merge(pd.read_csv('/share/mnch/anemia/code/reference/model/anemia_thresholds.csv'), on=["age_group_id", "sex_id", "pregnant"], how="left")
mean_and_sd_hgb

,location_id,sex_id,age_group_id,vehicle,draw,mean,sd,scenario,pregnant,age_group_name,grp,hgb_lower_mild,hgb_upper_mild,hgb_lower_moderate,hgb_upper_moderate,hgb_lower_severe,hgb_upper_severe,hgb_lower_anemic,hgb_upper_anemic
0,163,2,2,rice,draw_0,141.027303,11.056572,baseline,0,Early Neonatal,under_5,145,160,100,145,0,100,0,160
1,163,2,2,rice,draw_1,141.174892,15.419496,baseline,0,Early Neonatal,under_5,145,160,100,145,0,100,0,160
2,163,2,2,rice,draw_2,143.047093,12.937057,baseline,0,Early Neonatal,under_5,145,160,100,145,0,100,0,160
3,163,2,2,rice,draw_3,142.497768,14.108613,baseline,0,Early Neonatal,under_5,145,160,100,145,0,100,0,160
4,163,2,2,rice,draw_4,141.204487,13.020095,baseline,0,Early Neonatal,under_5,145,160,100,145,0,100,0,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91995,214,2,389,bouillon,draw_495,100.356215,13.412401,intervention,0,6-11 months,under_5,100,110,70,100,0,70,0,110
91996,214,2,389,bouillon,draw_496,100.399740,14.305163,intervention,0,6-11 months,under_5,100,110,70,100,0,70,0,110
91997,214,2,389,bouillon,draw_497,99.841120,15.815794,intervention,0,6-11 months,under_5,100,110,70,100,0,70,0,110
91998,214,2,389,bouillon,draw_498,100.311649,15.249249,intervention,0,6-11 months,under_5,100,110,70,100,0,70,0,110


In [28]:
assert (
    (mean_and_sd_hgb.hgb_upper_mild == mean_and_sd_hgb.hgb_upper_anemic).all() &
    (mean_and_sd_hgb.hgb_lower_severe == mean_and_sd_hgb.hgb_lower_anemic).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [29]:
assert (
    (mean_and_sd_hgb.hgb_lower_mild == mean_and_sd_hgb.hgb_upper_moderate).all() &
    (mean_and_sd_hgb.hgb_lower_moderate == mean_and_sd_hgb.hgb_upper_severe).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [30]:
def _hemoglobin_distribution_parts_from_mean_sd(mean, sd):
    # NOTE: This is an unusual ensemble distribution. We should add functionality to the
    # EnsembleDistribution class to make this easier.
    x_min = 0
    x_max = 220
    gamma_params = risk_distributions.risk_distributions.Gamma.get_parameters(
        mean=mean, sd=sd
    )
    # NOTE: We have to override these, otherwise Gamma is overly conservative in what values
    # are computable
    # https://github.com/ihmeuw/risk_distributions/issues/61
    gamma_params["x_min"] = x_min
    gamma_params["x_max"] = x_max
    hemoglobin_distribution_gamma_part = risk_distributions.risk_distributions.Gamma(
        gamma_params
    )

    # NOTE: Forced to duplicate https://github.com/ihmeuw/risk_distributions/blob/a9ed9d7e8372590018355012a7a7ffefa87b0819/src/risk_distributions/risk_distributions.py#L428-L434
    # because it doesn't permit the custom x_min and x_max, and these are used in calculating the others
    mgumbel_params = pd.DataFrame({
        "loc": x_max - mean - (np.euler_gamma * np.sqrt(6) / np.pi * sd),
        "scale": np.sqrt(6) / np.pi * sd,
        "x_min": x_min,
        "x_max": x_max,
    })
    hemoglobin_distribution_mgumbel_part = (
        risk_distributions.risk_distributions.MirroredGumbel(mgumbel_params)
    )
    return hemoglobin_distribution_gamma_part, hemoglobin_distribution_mgumbel_part

(
    hemoglobin_distribution_gamma_part,
    hemoglobin_distribution_mgumbel_part,
) = _hemoglobin_distribution_parts_from_mean_sd(mean_and_sd_hgb['mean'], mean_and_sd_hgb.sd)

def cdf(x):
    gamma_cdf = hemoglobin_distribution_gamma_part.cdf(x)
    # NOTE: There is a bug in this CDF function -- it is reversed!
    # https://github.com/ihmeuw/risk_distributions/issues/62
    mgumbel_cdf = 1 - hemoglobin_distribution_mgumbel_part.cdf(x)
    return (
        0.4
        * gamma_cdf
        + 0.6
        * mgumbel_cdf
    )

In [31]:
mean_and_sd_hgb["severe"] = cdf(mean_and_sd_hgb.hgb_upper_severe.copy()) - cdf(mean_and_sd_hgb.hgb_lower_severe.copy())
mean_and_sd_hgb["moderate"] = cdf(mean_and_sd_hgb.hgb_upper_moderate.copy()) - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["mild"] = cdf(mean_and_sd_hgb.hgb_upper_mild.copy()) - mean_and_sd_hgb["moderate"].copy() - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["anemic"] = mean_and_sd_hgb["mild"] + mean_and_sd_hgb["moderate"] + mean_and_sd_hgb["severe"]
mean_and_sd_hgb

,location_id,sex_id,age_group_id,vehicle,draw,mean,sd,scenario,pregnant,age_group_name,grp,hgb_upper_mild,hgb_upper_moderate,hgb_lower_severe,hgb_upper_severe,severe,moderate,mild,anemic
0,163,2,2,rice,draw_0,141.027303,11.056572,baseline,0,Early Neonatal,under_5,160,145,0,100,0.002889,0.610267,0.364111,0.977267
1,163,2,2,rice,draw_1,141.174892,15.419496,baseline,0,Early Neonatal,under_5,160,145,0,100,0.011504,0.555625,0.346515,0.913643
2,163,2,2,rice,draw_2,143.047093,12.937057,baseline,0,Early Neonatal,under_5,160,145,0,100,0.004745,0.520340,0.406191,0.931276
3,163,2,2,rice,draw_3,142.497768,14.108613,baseline,0,Early Neonatal,under_5,160,145,0,100,0.007197,0.529453,0.381172,0.917822
4,163,2,2,rice,draw_4,141.204487,13.020095,baseline,0,Early Neonatal,under_5,160,145,0,100,0.005880,0.578951,0.366934,0.951764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91995,214,2,389,bouillon,draw_495,100.356215,13.412401,intervention,0,6-11 months,under_5,110,100,0,70,0.020685,0.433465,0.307961,0.762110
91996,214,2,389,bouillon,draw_496,100.399740,14.305163,intervention,0,6-11 months,under_5,110,100,0,70,0.025494,0.428569,0.289861,0.743924
91997,214,2,389,bouillon,draw_497,99.841120,15.815794,intervention,0,6-11 months,under_5,110,100,0,70,0.036960,0.433323,0.262796,0.733078
91998,214,2,389,bouillon,draw_498,100.311649,15.249249,intervention,0,6-11 months,under_5,110,100,0,70,0.031518,0.426077,0.272561,0.730156


In [32]:
disability_weights = pd.read_hdf('/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf')
disability_weights = disability_weights[disability_weights.healthstate.isin(['anemia_mild', 'anemia_mod', 'anemia_sev'])].set_index('healthstate').filter(like='draw_')
disability_weights.columns.name = 'draw'
disability_weights = disability_weights.stack().rename('disability_weight').reset_index()
disability_weights

,healthstate,draw,disability_weight
0,anemia_mild,draw_0,0.002420
1,anemia_mild,draw_1,0.003172
2,anemia_mild,draw_2,0.002644
3,anemia_mild,draw_3,0.003085
4,anemia_mild,draw_4,0.001845
...,...,...,...
2995,anemia_sev,draw_995,0.174969
2996,anemia_sev,draw_996,0.086798
2997,anemia_sev,draw_997,0.131996
2998,anemia_sev,draw_998,0.124427


In [33]:
mean_and_sd_hgb = mean_and_sd_hgb.merge(
    disability_weights[disability_weights.healthstate == 'anemia_mild'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'mild_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_mod'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'moderate_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_sev'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'severe_dw'}),
    validate="m:1",
)

In [34]:
mean_and_sd_hgb["mild_ylds"] = mean_and_sd_hgb.mild * mean_and_sd_hgb.mild_dw
mean_and_sd_hgb["moderate_ylds"] = mean_and_sd_hgb.moderate * mean_and_sd_hgb.moderate_dw
mean_and_sd_hgb["severe_ylds"] = mean_and_sd_hgb.severe * mean_and_sd_hgb.severe_dw
mean_and_sd_hgb['anemic_ylds'] = mean_and_sd_hgb['mild_ylds'] + mean_and_sd_hgb['moderate_ylds'] + mean_and_sd_hgb['severe_ylds']

In [35]:
index_cols = ['location_id','age_group_id','sex_id','draw','vehicle']
value_cols = ["mild", "moderate", "severe", "anemic", "mild_ylds", "moderate_ylds", "severe_ylds", "anemic_ylds"]

baseline_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'baseline'].set_index(index_cols)[value_cols]
counterfactual_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'intervention'].set_index(index_cols)[value_cols]


In [36]:
baseline_anemia.sort_index()

mild  moderate  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.601897  0.378337   
                                draw_1   rice      0.732626  0.241402   
                                draw_10  rice      0.588778  0.404837   
                                draw_100 rice      0.746376  0.237778   
                                draw_101 rice      0.629293  0.354230   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.301774  0.466991   
                                draw_96  bouillon  0.271482  0.436388   
                                draw_97  bouillon  0.278500  0.433357   
                                draw_98  bouillon  0.300059  0.449278   
                                draw_99  bouillon  0.254778  0.410775   

                                                     severe    anemic  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.000131  0.980365   
                                draw_1   rice      0.000023  0.974051   
                                draw_10  rice      0.000034  0.993649   
                                draw_100 rice      0.000011  0.984164   
                                draw_101 rice      0.000073  0.983595   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.024565  0.793331   
                                draw_96  bouillon  0.033324  0.741193   
                                draw_97  bouillon  0.030039  0.741896   
                                draw_98  bouillon  0.024007  0.773344   
                                draw_99  bouillon  0.037163  0.702716   

                                                   mild_ylds  moderate_ylds  \
location_id age_group_id sex_id draw     vehicle                              
163         2            1      draw_0   rice       0.001457       0.022436   
                                draw_1   rice       0.002324       0.013777   
                                draw_10  rice       0.002025       0.017902   
                                draw_100 rice       0.002757       0.013343   
                                draw_101 rice       0.001283       0.015700   
...                                                      ...            ...   
214         389          2      draw_95  bouillon   0.001704       0.030524   
                                draw_96  bouillon   0.001429       0.021497   
                                draw_97  bouillon   0.000777       0.025708   
                                draw_98  bouillon   0.001174       0.017264   
                                draw_99  bouillon   0.001070       0.017122   

                                                   severe_ylds  anemic_ylds  
location_id age_group_id sex_id draw     vehicle                             
163         2            1      draw_0   rice         0.000026     0.023919  
                                draw_1   rice         0.000004     0.016105  
                                draw_10  rice         0.000005     0.019932  
                                draw_100 rice         0.000002     0.016103  
                                draw_101 rice         0.000012     0.016994  
...                                                        ...          ...  
214         389          2      draw_95  bouillon     0.004190     0.036417  
                                draw_96  bouillon     0.004681     0.027608  
                                draw_97  bouillon     0.006003     0.032488  
                                draw_98  bouillon     0.002181     0.020619  
                                draw_99  bouillon     0.004414     0.022606  

[46000 rows x 8 columns]

In [37]:
counterfactual_anemia.sort_index()

mild  moderate  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.625048  0.350675   
                                draw_1   rice      0.749464  0.217271   
                                draw_10  rice      0.621153  0.370919   
                                draw_100 rice      0.767513  0.211995   
                                draw_101 rice      0.653749  0.325807   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.304953  0.438893   
                                draw_96  bouillon  0.272010  0.412866   
                                draw_97  bouillon  0.278902  0.408987   
                                draw_98  bouillon  0.301852  0.421879   
                                draw_99  bouillon  0.253988  0.389590   

                                                     severe    anemic  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.000119  0.975841   
                                draw_1   rice      0.000021  0.966756   
                                draw_10  rice      0.000031  0.992102   
                                draw_100 rice      0.000010  0.979518   
                                draw_101 rice      0.000066  0.979622   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.021857  0.765703   
                                draw_96  bouillon  0.029951  0.714827   
                                draw_97  bouillon  0.026952  0.714841   
                                draw_98  bouillon  0.021402  0.745133   
                                draw_99  bouillon  0.033601  0.677179   

                                                   mild_ylds  moderate_ylds  \
location_id age_group_id sex_id draw     vehicle                              
163         2            1      draw_0   rice       0.001513       0.020795   
                                draw_1   rice       0.002378       0.012400   
                                draw_10  rice       0.002136       0.016402   
                                draw_100 rice       0.002836       0.011896   
                                draw_101 rice       0.001333       0.014440   
...                                                      ...            ...   
214         389          2      draw_95  bouillon   0.001722       0.028687   
                                draw_96  bouillon   0.001432       0.020339   
                                draw_97  bouillon   0.000778       0.024262   
                                draw_98  bouillon   0.001181       0.016212   
                                draw_99  bouillon   0.001067       0.016239   

                                                   severe_ylds  anemic_ylds  
location_id age_group_id sex_id draw     vehicle                             
163         2            1      draw_0   rice         0.000024     0.022332  
                                draw_1   rice         0.000003     0.014781  
                                draw_10  rice         0.000004     0.018543  
                                draw_100 rice         0.000002     0.014734  
                                draw_101 rice         0.000011     0.015783  
...                                                        ...          ...  
214         389          2      draw_95  bouillon     0.003728     0.034137  
                                draw_96  bouillon     0.004208     0.025978  
                                draw_97  bouillon     0.005386     0.030426  
                                draw_98  bouillon     0.001944     0.019337  
                                draw_99  bouillon     0.003991     0.021297  

[46000 rows x 8 columns]

In [38]:
baseline_anemia - counterfactual_anemia

mild  moderate  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice     -0.023150  0.027662   
                                draw_1   rice     -0.016838  0.024131   
                                draw_10  rice     -0.032375  0.033918   
                                draw_100 rice     -0.021138  0.025783   
                                draw_101 rice     -0.024456  0.028423   
...                                                     ...       ...   
214         389          2      draw_95  bouillon -0.003179  0.028098   
                                draw_96  bouillon -0.000528  0.023521   
                                draw_97  bouillon -0.000401  0.024369   
                                draw_98  bouillon -0.001793  0.027399   
                                draw_99  bouillon  0.000790  0.021186   

                                                     severe    anemic  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.000012  0.004524   
                                draw_1   rice      0.000002  0.007295   
                                draw_10  rice      0.000004  0.001547   
                                draw_100 rice      0.000001  0.004646   
                                draw_101 rice      0.000007  0.003973   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.002708  0.027627   
                                draw_96  bouillon  0.003373  0.026366   
                                draw_97  bouillon  0.003088  0.027056   
                                draw_98  bouillon  0.002605  0.028211   
                                draw_99  bouillon  0.003562  0.025537   

                                                   mild_ylds  moderate_ylds  \
location_id age_group_id sex_id draw     vehicle                              
163         2            1      draw_0   rice      -0.000056       0.001640   
                                draw_1   rice      -0.000053       0.001377   
                                draw_10  rice      -0.000111       0.001500   
                                draw_100 rice      -0.000078       0.001447   
                                draw_101 rice      -0.000050       0.001260   
...                                                      ...            ...   
214         389          2      draw_95  bouillon  -0.000018       0.001837   
                                draw_96  bouillon  -0.000003       0.001159   
                                draw_97  bouillon  -0.000001       0.001446   
                                draw_98  bouillon  -0.000007       0.001053   
                                draw_99  bouillon   0.000003       0.000883   

                                                    severe_ylds  anemic_ylds  
location_id age_group_id sex_id draw     vehicle                              
163         2            1      draw_0   rice      2.408696e-06     0.001587  
                                draw_1   rice      3.980219e-07     0.001324  
                                draw_10  rice      5.228573e-07     0.001389  
                                draw_100 rice      2.200242e-07     0.001369  
                                draw_101 rice      1.152477e-06     0.001211  
...                                                         ...          ...  
214         389          2      draw_95  bouillon  4.618766e-04     0.002280  
                                draw_96  bouillon  4.737977e-04     0.001630  
                                draw_97  bouillon  6.170143e-04     0.002062  
                                draw_98  bouillon  2.366259e-04     0.001282  
                                draw_99  bouillon  4.230699e-04     0.001309  

[46000 rows x 8 columns]

In [39]:
# BUT our counterfactual is only in a world where everyone is iron-responsive.
iron_responsive_anemia_sequelae = [1004, 1005, 1006, 1008, 1009, 1010, 1012, 1013, 
                                   1014, 1016, 1017, 1018, 1020, 1021, 1022, 1024, 1025, 1026, 
                                   1028, 1029, 1030, 1032, 1033, 1034, 1361, 1364, 1367, 1373, 1376, 
                                   1379, 1385, 1388, 1391, 1397, 1400, 1403, 1409, 1412, 1415, 1421, 
                                   1424, 1427, 1433, 1436, 1439, 1445, 1448, 1451, 5213, 5216, 5219, 
                                   5222, 5225, 5228, 5237, 5240, 5243, 5246, 5249, 5252, 5261, 5264, 
                                   5267, 5270, 5273, 5276, 4985, 4988, 4991, 4994, 4997, 5000, 5009, 
                                   5012, 5015, 5678, 5681, 5684, 7214, 7217, 7220, 4952, 4955, 4958, 
                                   4961, 4964, 4967, 4976, 4979, 4982, 5627, 5630, 5633, 7202, 7205, 
                                   7208, 5393, 5396, 5399, 182, 183, 184, 240, 241, 242, 177, 178, 
                                   179, 144,145,146,172,173,174,525,526,527,1106,1107,1108,537,538,
                                   539,206,207,208, 22989, 22990, 22991, 22992, 22993, 22999, 23000, 
                                   23001, 23002, 23003, 23009, 23010, 23011, 23012, 23013,
                                   5567, 5570, 5573, 5579, 5582, 5585,
                                   23030, 23031, 23032, 23034, 23035, 23036, 23038, 23039, 23040,
                                   23042, 23043, 23044, 23046, 23047, 23048]

ira_prev = get_draws('sequela_id', iron_responsive_anemia_sequelae, 
                 source='como',
                 location_id=location_ids, 
                 age_group_id=age_group_ids,
                 sex_id=sex_ids,
                 year_id=2021,
                 release_id=9)
ira_prev = ira_prev.groupby(['location_id','sex_id','age_group_id'], as_index=False).sum().set_index(["location_id", "age_group_id", "sex_id"]).filter(like="draw")
ira_prev

/ihme/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/5000_analyze_results/.venv/lib/python3.11/site-packages/get_draws/api.py:192: UserWarning: No version_id was specified, so get_draws will automatically determine a best version ID to use from the given parameters. If you want to retrieve a specific version please pass a version_id directly.
  warnings.warn(


draw_0    draw_1   draw_10  draw_100  \
location_id age_group_id sex_id                                           
163         2            1       0.856356  0.853869  0.877692  0.903028   
            3            1       0.554237  0.615242  0.582226  0.579728   
            6            1       0.474995  0.430834  0.457424  0.499833   
            7            1       0.171226  0.141813  0.149330  0.167503   
            8            1       0.195740  0.169052  0.233220  0.202878   
...                                   ...       ...       ...       ...   
214         31           2       0.482638  0.487489  0.544101  0.456644   
            32           2       0.686516  0.666819  0.692734  0.659024   
            235          2       0.760906  0.622712  0.678574  0.668662   
            388          2       0.536408  0.572838  0.593318  0.569228   
            389          2       0.634429  0.584234  0.617535  0.591367   

                                 draw_101  draw_102  draw_103  draw_104  \
location_id age_group_id sex_id                                           
163         2            1       0.841963  0.870481  0.847444  0.900252   
            3            1       0.576149  0.506180  0.544890  0.555005   
            6            1       0.572597  0.483333  0.370659  0.395394   
            7            1       0.151381  0.149642  0.148790  0.208099   
            8            1       0.203240  0.198437  0.172822  0.213700   
...                                   ...       ...       ...       ...   
214         31           2       0.447244  0.387006  0.415944  0.573148   
            32           2       0.585745  0.694039  0.678205  0.574112   
            235          2       0.660820  0.702258  0.690640  0.671979   
            388          2       0.538080  0.569507  0.566607  0.588122   
            389          2       0.657035  0.666057  0.653659  0.631320   

                                 draw_105  draw_106  ...   draw_90   draw_91  \
location_id age_group_id sex_id                      ...                       
163         2            1       0.866400  0.912991  ...  0.909161  0.820569   
            3            1       0.556569  0.564146  ...  0.570062  0.566841   
            6            1       0.503090  0.468897  ...  0.393211  0.431615   
            7            1       0.152648  0.164651  ...  0.183739  0.166066   
            8            1       0.202767  0.204134  ...  0.149097  0.185008   
...                                   ...       ...  ...       ...       ...   
214         31           2       0.365575  0.472977  ...  0.383674  0.349002   
            32           2       0.676987  0.687330  ...  0.647978  0.675958   
            235          2       0.712695  0.665327  ...  0.731943  0.611530   
            388          2       0.621504  0.610280  ...  0.571198  0.550125   
            389          2       0.593248  0.653903  ...  0.625606  0.620183   

                                  draw_92   draw_93   draw_94   draw_95  \
location_id age_group_id sex_id                                           
163         2            1       0.887391  0.834679  0.854281  0.895827   
            3            1       0.541740  0.533796  0.520728  0.557474   
            6            1       0.417674  0.527071  0.519116  0.465155   
            7            1       0.196186  0.210944  0.183742  0.176057   
            8            1       0.180790  0.178806  0.193892  0.221681   
...                                   ...       ...       ...       ...   
214         31           2       0.587723  0.470768  0.582429  0.433256   
            32           2       0.554284  0.623295  0.659139  0.683349   
            235          2       0.635602  0.605795  0.622244  0.658268   
            388          2       0.572374  0.573902  0.595246  0.642702   
            389          2       0.604680  0.656069  0.613173  0.608834   

                                  draw_96   draw_97   draw_98   dr

In [40]:
ira_prev.columns.name = "draw"
ira_prev = ira_prev.stack().pipe(lambda s: s[s.index.get_level_values("draw").isin(DRAWS)])
ira_prev

location_id  age_group_id  sex_id  draw    
163          2             1       draw_0      0.856356
                                   draw_1      0.853869
                                   draw_10     0.877692
                                   draw_100    0.903028
                                   draw_101    0.841963
                                                 ...   
214          389           2       draw_95     0.608834
                                   draw_96     0.606151
                                   draw_97     0.628402
                                   draw_98     0.616976
                                   draw_99     0.621221
Length: 46000, dtype: float64

In [41]:
iron_responsive_proportion = ira_prev / baseline_anemia.anemic.droplevel("vehicle")
iron_responsive_proportion

location_id  age_group_id  sex_id  draw    
163          2             1       draw_0      0.873507
                                   draw_1      0.876616
                                   draw_10     0.883302
                                   draw_100    0.917558
                                   draw_101    0.856005
                                                 ...   
214          389           2       draw_95     0.767440
                                   draw_96     0.817805
                                   draw_97     0.847022
                                   draw_98     0.797803
                                   draw_99     0.884029
Length: 46000, dtype: float64

In [42]:
# NOTE: I am pretty sure this is correct, but it is quite difficult to think through *why*.

# First, observe that people in the population who start as non-anemic never factor into
# any of these metrics. If they started non-anemic, our counterfactual can only shift them up,
# so they did not change anemia categories between scenarios and hence have no importance to
# anemia prevalence or YLDs.

# So you can think of our hemoglobin distributions as only being of interest in the part
# of them below the anemia threshold.

# *Within* this subpopulation, we make the assumption that iron-responsive and non-iron-responsive
# anemic people have the same distributions of hemoglobin. This is probably not true, but GBD doesn't
# give us anything better.

# So you can think of our original distribution as a mixture of two parts, which are the same.
# Then we shift one of those parts (the iron-responsive part) and calculate all these stats from
# that new distribution.

# Our *actual* result should be about a mixture distribution that has the non-iron-responsive part
# the same as in baseline, with the shifted iron-responsive part.
# For all these metrics, it is pretty straightforward to see that the metric in such a mixture is
# just a weighted average of the metrics in each part.

counterfactual_anemia_accounting_for_non_response = (
    counterfactual_anemia.mul(iron_responsive_proportion, axis=0) +
    baseline_anemia.mul(1 - iron_responsive_proportion, axis=0)
)
counterfactual_anemia_accounting_for_non_response

mild  moderate  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.622119  0.354174   
                                draw_1   rice      0.747387  0.220249   
                                draw_10  rice      0.617375  0.374877   
                                draw_100 rice      0.765771  0.214120   
                                draw_101 rice      0.650228  0.329900   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.304213  0.445428   
                                draw_96  bouillon  0.271913  0.417152   
                                draw_97  bouillon  0.278840  0.412715   
                                draw_98  bouillon  0.301489  0.427419   
                                draw_99  bouillon  0.254080  0.392046   

                                                     severe    anemic  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.000120  0.976413   
                                draw_1   rice      0.000021  0.967656   
                                draw_10  rice      0.000031  0.992283   
                                draw_100 rice      0.000010  0.979901   
                                draw_101 rice      0.000067  0.980194   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.022487  0.772128   
                                draw_96  bouillon  0.030566  0.719631   
                                draw_97  bouillon  0.027424  0.718980   
                                draw_98  bouillon  0.021929  0.750837   
                                draw_99  bouillon  0.034014  0.680141   

                                                   mild_ylds  moderate_ylds  \
location_id age_group_id sex_id draw     vehicle                              
163         2            1      draw_0   rice       0.001506       0.021003   
                                draw_1   rice       0.002371       0.012570   
                                draw_10  rice       0.002123       0.016578   
                                draw_100 rice       0.002829       0.012016   
                                draw_101 rice       0.001326       0.014621   
...                                                      ...            ...   
214         389          2      draw_95  bouillon   0.001718       0.029114   
                                draw_96  bouillon   0.001431       0.020550   
                                draw_97  bouillon   0.000778       0.024483   
                                draw_98  bouillon   0.001180       0.016424   
                                draw_99  bouillon   0.001067       0.016341   

                                                   severe_ylds  anemic_ylds  
location_id age_group_id sex_id draw     vehicle                             
163         2            1      draw_0   rice         0.000024     0.022532  
                                draw_1   rice         0.000003     0.014944  
                                draw_10  rice         0.000004     0.018705  
                                draw_100 rice         0.000002     0.014846  
                                draw_101 rice         0.000011     0.015958  
...                                                        ...          ...  
214         389          2      draw_95  bouillon     0.003835     0.034667  
                                draw_96  bouillon     0.004294     0.026275  
                                draw_97  bouillon     0.005481     0.030742  
                                draw_98  bouillon     0.001992     0.019596  
                                draw_99  bouillon     0.004040     0.021448  

[46000 rows x 8 columns]

In [43]:
averted_anemia = baseline_anemia - counterfactual_anemia_accounting_for_non_response
averted_anemia

mild  moderate  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice     -0.020222  0.024163   
                                draw_1   rice     -0.014761  0.021154   
                                draw_10  rice     -0.028597  0.029960   
                                draw_100 rice     -0.019395  0.023657   
                                draw_101 rice     -0.020935  0.024330   
...                                                     ...       ...   
214         389          2      draw_95  bouillon -0.002440  0.021564   
                                draw_96  bouillon -0.000432  0.019236   
                                draw_97  bouillon -0.000340  0.020641   
                                draw_98  bouillon -0.001431  0.021859   
                                draw_99  bouillon  0.000698  0.018729   

                                                     severe    anemic  \
location_id age_group_id sex_id draw     vehicle                        
163         2            1      draw_0   rice      0.000011  0.003952   
                                draw_1   rice      0.000002  0.006395   
                                draw_10  rice      0.000003  0.001366   
                                draw_100 rice      0.000001  0.004263   
                                draw_101 rice      0.000006  0.003401   
...                                                     ...       ...   
214         389          2      draw_95  bouillon  0.002078  0.021202   
                                draw_96  bouillon  0.002758  0.021563   
                                draw_97  bouillon  0.002615  0.022917   
                                draw_98  bouillon  0.002078  0.022507   
                                draw_99  bouillon  0.003149  0.022575   

                                                      mild_ylds  \
location_id age_group_id sex_id draw     vehicle                  
163         2            1      draw_0   rice     -4.894694e-05   
                                draw_1   rice     -4.682720e-05   
                                draw_10  rice     -9.833370e-05   
                                draw_100 rice     -7.165550e-05   
                                draw_101 rice     -4.268624e-05   
...                                                         ...   
214         389          2      draw_95  bouillon -1.377427e-05   
                                draw_96  bouillon -2.271269e-06   
                                draw_97  bouillon -9.477678e-07   
                                draw_98  bouillon -5.598371e-06   
                                draw_99  bouillon  2.932753e-06   

                                                   moderate_ylds  \
location_id age_group_id sex_id draw     vehicle                   
163         2            1      draw_0   rice           0.001433   
                                draw_1   rice           0.001207   
                                draw_10  rice           0.001325   
                                draw_100 rice           0.001328   
                                draw_101 rice           0.001078   
...                                                          ...   
214         389          2      draw_95  bouillon       0.001409   
                                draw_96  bouillon       0.000948   
                                draw_97  bouillon       0.001224   
                                draw_98  bouillon       0.000840   
                                draw_99  bouillon       0.000781   

                                                    severe_ylds  anemic_ylds  
location_id age_group_id sex_id draw     vehicle                              
163         2            1      draw_0   rice      2.104013e-06     0.001386  
                                draw_1   rice      3.489124e-07     0.001161  
                                draw_10  rice      4.618407e-07     0.001227  
   

In [44]:
assert (averted_anemia.anemic_ylds > 0).all()

In [45]:
pop = (
    db_queries.get_population(age_group_ids, location_id=location_ids, year_id=2030, sex_id=sex_ids, release_id=9, forecasted_pop=True)
        .set_index(["location_id", "age_group_id", "sex_id"]).population
)
pop

location_id  age_group_id  sex_id
163          2             1         1.951231e+05
             3             1         5.804971e+05
             6             1         5.481399e+07
             7             1         5.950935e+07
             8             1         6.380942e+07
                                         ...     
214          20            2         8.610161e+05
             30            2         3.751797e+05
             31            2         2.051265e+05
             32            2         6.935082e+04
             235           2         2.227497e+04
Name: population, Length: 84, dtype: float64

In [46]:
ylds = pd.concat([
    (baseline_anemia.anemic_ylds.unstack("draw").mean(axis=1) * pop).rename("value").reset_index().assign(scenario="baseline"),
    (counterfactual_anemia_accounting_for_non_response.anemic_ylds.unstack("draw").mean(axis=1) * pop).rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
ylds

,location_id,age_group_id,sex_id,vehicle,value,scenario
0,163,2,1,rice,3758.519452,baseline
1,163,2,2,rice,5611.317797,baseline
2,163,3,1,rice,9622.039914,baseline
3,163,3,2,rice,9932.576605,baseline
4,163,6,1,rice,771405.590951,baseline
...,...,...,...,...,...,...
179,214,235,2,bouillon,1136.562150,intervention
180,214,388,1,bouillon,NaN,intervention
181,214,388,2,bouillon,NaN,intervention
182,214,389,1,bouillon,NaN,intervention


In [49]:
import pathlib

for location_id in ylds.location_id.unique():
    location_names = {
        163: 'india',
        214: 'nigeria'
    }
    assert location_id in location_names
    path = f'./{location_names[location_id]}/ylds.parquet'
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    ylds[ylds.location_id == location_id].to_parquet(path)

In [ ]:
# TODO: Save out anemia prevalence, in sim format (?)